# Urban Transit & BRT Systems: Africa–USA Analysis

**Domain:** Global Transportation  |  **Focus:** Africa and the USA

Rapid urbanisation in Africa (≈3.3–3.5% urban population growth per year) and the USA's ageing transit infrastructure create urgent demand for efficient, data-driven public transit. Bus Rapid Transit (BRT) offers a cost-effective alternative to rail.

This notebook explores:
1. **BRT system performance** – ridership, corridor length, stations and fleet (ITDP Global BRT Data, https://brtdata.org/).
2. **Urban growth drivers** – World Bank urban population, urban growth rate and urbanisation share.
3. **Economic context** – GDP per capita versus BRT ridership.

Data is read from `data/raw/` (newest timestamped snapshot written by `src/ingestion/ingest.py`, falling back to the canonical CSVs).

In [ ]:
import re
import sqlite3
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (11, 6)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
CONTINENT_COLORS = {"Africa": "#E07A2E", "North America": "#2E6FE0"}
print("Libraries loaded; sqlite3 version", sqlite3.sqlite_version)

In [ ]:
# Locate data/raw relative to the notebook (works from notebooks/ or the project root)
CANDIDATES = [Path("../data/raw"), Path("../../data/raw"), Path("data/raw")]
RAW_DIR = next(p for p in CANDIDATES if (p / "brt_systems_africa_usa.csv").exists())


def latest_raw(stem):
    """Return the newest timestamped snapshot of a dataset, else the canonical CSV."""
    pattern = re.compile(rf"^{stem}_\d{{8}}T\d{{6}}Z\.csv$")
    snaps = sorted(p for p in RAW_DIR.glob(f"{stem}_*.csv") if pattern.match(p.name))
    return snaps[-1] if snaps else RAW_DIR / f"{stem}.csv"


brt = pd.read_csv(latest_raw("brt_systems_africa_usa"))
wb_files = {
    "urban_population": "wb_urban_population",
    "urban_growth_rate_pct": "wb_urban_growth_rate",
    "gdp_per_capita_usd": "wb_gdp_per_capita",
    "urban_pct_of_total": "wb_urban_pct_total",
}
wb = None
for value_col, stem in wb_files.items():
    path = latest_raw(stem)
    df = pd.read_csv(path)[["iso3", "country_name", "year", value_col]]
    print(f"{path.name:<45} {len(df):>4} rows")
    wb = df if wb is None else wb.merge(df, on=["iso3", "country_name", "year"], how="outer")
wb["year"] = wb["year"].astype(int)
wb["region"] = wb["iso3"].map(lambda c: "USA" if c == "USA" else "Africa")

# Latest World Bank values per country, joined to each BRT system
latest_wb = wb.sort_values("year").groupby("iso3").tail(1)
brt_wb = brt.merge(latest_wb.drop(columns=["country_name", "region"]), on="iso3", how="left")
brt_wb["ridership_per_km"] = brt_wb["daily_ridership"] / brt_wb["corridor_length_km"]
brt_wb["ridership_per_station"] = brt_wb["daily_ridership"] / brt_wb["num_stations"]
brt_wb["fleet_utilization_ratio"] = brt_wb["daily_ridership"] / brt_wb["fleet_size"]
print(f"BRT systems: {len(brt)} | WB country-years: {len(wb)} ({wb['iso3'].nunique()} countries)")
print(f"BRT systems matched to WB indicators: {brt_wb['year'].notna().sum()}/{len(brt_wb)}")

In [ ]:
print("BRT shape:", brt.shape, "| World Bank shape:", wb.shape)
print("\nBRT dtypes:\n", brt.dtypes, sep="")
print("\nMissing values (BRT):", int(brt.isna().sum().sum()), "| (World Bank):", int(wb.isna().sum().sum()))
display(brt.describe().T)
display(wb.describe().T)

## 1. BRT Systems Analysis

How do African and US BRT systems compare in scale and productivity? US networks such as LA Metro Rapid and NYC Select Bus Service are long, route-based networks, while African systems are compact, high-intensity trunk corridors.

In [ ]:
top10 = brt.nlargest(10, "daily_ridership").copy()
top10["label"] = top10["system_name"] + " (" + top10["city"] + ")"
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=top10, y="label", x="daily_ridership", hue="continent",
            palette=CONTINENT_COLORS, dodge=False, ax=ax)
for patch, value in zip(ax.patches, top10["daily_ridership"]):
    ax.text(patch.get_width() + 4000, patch.get_y() + patch.get_height() / 2,
            f"{value:,.0f}", va="center", fontsize=9)
ax.set_title("Top 10 BRT Systems by Daily Ridership (Africa & USA)", fontsize=14, weight="bold")
ax.set_xlabel("Daily ridership (passengers/day)")
ax.set_ylabel("")
ax.legend(title="Continent", loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
fig = px.scatter(
    brt, x="corridor_length_km", y="daily_ridership", color="continent",
    size="fleet_size", hover_name="system_name",
    hover_data={"city": True, "num_stations": True, "year_opened": True},
    log_x=True, color_discrete_map=CONTINENT_COLORS,
    title="Corridor Length vs Daily Ridership (bubble size = fleet size, log x-axis)",
    labels={"corridor_length_km": "Corridor length (km, log scale)", "daily_ridership": "Daily ridership"},
)
fig.update_layout(height=550)
fig.show()
print(brt.groupby("continent")[["corridor_length_km", "daily_ridership"]].median())

## 2. Urban Growth Analysis

World Bank indicators SP.URB.GROW (urban population growth, annual %) and SP.URB.TOTL (urban population) show where transit demand is accelerating fastest.

In [ ]:
latest_year = int(wb["year"].max())
africa_latest = wb[(wb["region"] == "Africa") & (wb["year"] == latest_year)].dropna(subset=["urban_growth_rate_pct"])
top15 = africa_latest.nlargest(15, "urban_growth_rate_pct")
brt_iso = set(brt["iso3"])
colors = ["#E07A2E" if c in brt_iso else "#9AA5B1" for c in top15["iso3"]]

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top15["country_name"], top15["urban_growth_rate_pct"], color=colors)
ax.invert_yaxis()
africa_avg = africa_latest["urban_growth_rate_pct"].mean()
ax.axvline(africa_avg, color="black", ls="--", lw=1, label=f"Africa average ({africa_avg:.2f}%)")
usa_rate = wb.loc[(wb["iso3"] == "USA") & (wb["year"] == latest_year), "urban_growth_rate_pct"]
if not usa_rate.empty:
    ax.axvline(usa_rate.iloc[0], color="#2E6FE0", ls=":", lw=1.5, label=f"USA ({usa_rate.iloc[0]:.2f}%)")
ax.set_title(f"Top 15 African Countries by Urban Population Growth Rate ({latest_year})\n(orange = operates BRT)",
             fontsize=13, weight="bold")
ax.set_xlabel("Urban population growth (annual %)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
top5_africa = (wb[(wb["region"] == "Africa") & (wb["year"] == latest_year)]
               .nlargest(5, "urban_population")["iso3"].tolist())
selected = top5_africa + (["USA"] if "USA" in set(wb["iso3"]) else [])
trend = wb[wb["iso3"].isin(selected)].copy()
trend["urban_population_m"] = trend["urban_population"] / 1e6

fig, ax = plt.subplots(figsize=(11, 6))
for country, grp in trend.sort_values("year").groupby("country_name"):
    style = "--" if grp["iso3"].iloc[0] == "USA" else "-"
    ax.plot(grp["year"], grp["urban_population_m"], style, marker="o", lw=2, label=country)
ax.set_title("Urban Population Trend: Top 5 African Countries + USA", fontsize=14, weight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Urban population (millions)")
ax.set_xticks(sorted(trend["year"].unique()))
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()
if "USA" not in selected:
    print("Note: the loaded World Bank snapshot has no USA rows; run the ingestion stage to refresh.")

## 3. GDP and Transit Investment

Does national wealth predict BRT usage? Each BRT system is joined (by ISO3) to its country's latest GDP per capita (NY.GDP.PCAP.CD).

In [ ]:
gdp_plot = brt_wb.dropna(subset=["gdp_per_capita_usd"])
fig = px.scatter(
    gdp_plot, x="gdp_per_capita_usd", y="daily_ridership", color="continent",
    size="corridor_length_km", hover_name="system_name", hover_data=["city", "ridership_per_km"],
    log_x=True, color_discrete_map=CONTINENT_COLORS,
    title="GDP per Capita vs BRT Daily Ridership (bubble size = corridor km)",
    labels={"gdp_per_capita_usd": "GDP per capita (current US$, log scale)", "daily_ridership": "Daily ridership"},
)
fig.update_layout(height=550)
fig.show()
print(f"Systems with GDP data: {len(gdp_plot)}/{len(brt_wb)}")
print(gdp_plot.groupby("continent")[["gdp_per_capita_usd", "ridership_per_km"]].mean())

In [ ]:
numeric_cols = ["year_opened", "corridor_length_km", "daily_ridership", "num_stations", "fleet_size",
                "ridership_per_km", "ridership_per_station", "fleet_utilization_ratio",
                "gdp_per_capita_usd", "urban_growth_rate_pct", "urban_pct_of_total"]
corr = brt_wb[numeric_cols].dropna(axis=1, how="all").corr()
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.75}, ax=ax)
ax.set_title("Correlation Matrix of BRT Features and Country Indicators", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

## 4. Key Insights and Conclusions

* **African BRT corridors are far more intensely used.** Lagos BRT-Lite (≈200k riders/day on 22 km) and Dar es Salaam DART (≈180k on 20.9 km) carry 8,600–9,100 riders per corridor-km — roughly 7x the US average of ≈1,200.
* **US systems win on absolute scale, not intensity.** NYC Select Bus Service (400k/day) and LA Metro Rapid (350k/day) top the ridership chart but run on 320–580 km networks.
* **Fleet utilisation is highest in Africa.** DART and Addis Ababa BRT exceed 1,000 daily riders per vehicle, signalling capacity pressure and a case for fleet expansion.
* **Urban growth keeps demand rising.** African urban populations grow ≈3.3% a year on average versus ≈0.6% in the USA; Tanzania, Uganda and Ethiopia — all BRT countries — grow faster than 4.5%.
* **Wealth does not drive ridership.** GDP per capita is negatively correlated with ridership per km in this sample (r ≈ -0.55): the lowest-income BRT cities show the highest riders/km.
* **Africa's BRT wave is recent.** 5 of the 10 African systems opened in 2022–2024, while most US systems date from 1983–2016.

The summary table below consolidates these figures by continent.

In [ ]:
summary = brt_wb.groupby("continent").agg(
    systems=("system_name", "count"),
    total_daily_ridership=("daily_ridership", "sum"),
    median_daily_ridership=("daily_ridership", "median"),
    total_corridor_km=("corridor_length_km", "sum"),
    avg_ridership_per_km=("ridership_per_km", "mean"),
    avg_ridership_per_station=("ridership_per_station", "mean"),
    avg_riders_per_vehicle=("fleet_utilization_ratio", "mean"),
    median_year_opened=("year_opened", "median"),
    avg_gdp_per_capita_usd=("gdp_per_capita_usd", "mean"),
    avg_urban_growth_pct=("urban_growth_rate_pct", "mean"),
).round(1)
display(summary.T)

region_growth = wb.groupby(["region", "year"])["urban_growth_rate_pct"].mean().unstack(0).round(2)
print("Average urban population growth rate (%) by region and year:")
display(region_growth)